# PRÁCTICA 3 SEMANA 1

**Se recomienda ejecutar los códigos de este jupyer notebook en la misma carpeta donde reside dicho documento, debido a la dependencia de ciertos documentos para el correcto uso del mismo.**

En primer lugar, vamos a importar las librerías necesarias y vamos a importar los csv necesarios.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
import scipy.stats as stats
from sklearn.metrics import r2_score
from sklearn.dummy import DummyClassifier
from scipy.stats import zscore
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegressionCV

In [ ]:
explotacion = pd.read_csv("fuga_clientes_empresa_telefonica_explotacion.csv")
construccion = pd.read_csv("fuga_clientes_empresa_telefonica_construccion.csv")

En esta semana 1 vamos a centrarnos en la limpieza y la creación de un modelo dummy y un modelo de regresión lineal con sklearn completando la información con las prácticas de semanas anteriores. Tenemos que especificar que el dataframe de entrenamiento es construcción y el de testeo es el de expliotación, ya que el de explotación en la variable a predecir tiene valores Nan.

In [ ]:
explotacion

In [ ]:
construccion

#### PRIMERA PARTE
En primer lugar, vamos a centrarnos en la limpieza de los dos datasets. Para ello, vamos a centrarnos primeramente en eliminar las variables colineales, redefiniendo los datasets sin la variable target(Churn Status), ya que es la variable a predecir y es la variable más dependiente. Después vamos a eliminar los outliers y por último redefinir las variables si hicieran falta en caso de que halla valores nulos o outliers.

In [ ]:
# Utilizamos str.extract para eliminar los strings de la id para que no nos de problema
explotacion["Customer ID"] = explotacion["Customer ID"].str.extract(r"(\d+)")
explotacion["Customer ID"] = explotacion["Customer ID"].astype(int)
construccion["Customer ID"] = construccion["Customer ID"].str.extract(r"(\d+)")
construccion["Customer ID"] = construccion["Customer ID"].astype(int)

In [ ]:
explotacion_mod = explotacion.drop(columns=["Churn Status"])
construccion_mod = construccion.drop(columns=["Churn Status"])

Redefinimos el ID. Desarrollamos las columnas de network y most usando get_dummies, ya que son variables categóricas para calcular la covarianza. Con la matriz de covarianza podremos hallar su rango y por tanto las variables colineales. Hacemos lo mismo para los dos dataframes. Pero antes vamos a concatenar ambos dataframes para que se eliminen las mismas variables en ambos dataframes. Es una medida de seguridad

In [ ]:

# Añadimos un identificador temporal para poder separar después del get_dummies
explotacion_mod['_dataset'] = 'explotacion'
construccion_mod['_dataset'] = 'construccion'

# Concatenamos los dataframes
combined = pd.concat([construccion_mod, explotacion_mod], axis=0, ignore_index=True)

# Aplicar get_dummies a todo el dataset concatenado
# Así se crean las mismas columnas para ambos datasets
combined = pd.get_dummies(combined, columns=["Network type subscription in Month 1"], drop_first=True)
combined = pd.get_dummies(combined, columns=["Network type subscription in Month 2"], drop_first=True)
combined = pd.get_dummies(combined, columns=["Most Loved Competitor network in Month 1"], drop_first=True)
combined = pd.get_dummies(combined, columns=["Most Loved Competitor network in Month 2"], drop_first=True)

# Separamos de nuevo en construccion y explotacion
construccion_mod = combined[combined['_dataset'] == 'construccion'].copy()
explotacion_mod = combined[combined['_dataset'] == 'explotacion'].copy()

# Eliminamos el identificador temporal
construccion_mod = construccion_mod.drop(columns=['_dataset'])
explotacion_mod = explotacion_mod.drop(columns=['_dataset'])

# Verificar que ahora tienen el mismo número de columnas
print(f"Construcción: {construccion_mod.shape[1]} columnas")
print(f"Explotación: {explotacion_mod.shape[1]} columnas")

# Calculamos matrices de covarianza
cov_construccion = construccion_mod.cov()
cov_explotacion = explotacion_mod.cov()


In [ ]:
explotacion_mod

In [ ]:
construccion_mod

In [ ]:
cov_explotacion

In [ ]:
cov_construccion

Para eliminar las variables colineales vamos a hallar las matrices de covarianza y sus rangos. Con ello podremos ver cuantas variables dependientes o colineales hay.

In [ ]:
rango_construccion = np.linalg.matrix_rank(cov_construccion)
rango_explotacion = np.linalg.matrix_rank(cov_explotacion)
print(f"Explotación:{cov_explotacion.shape}, Construcción:{cov_construccion.shape}")
print(f"Explotación: {rango_explotacion}, Construcción:{rango_construccion}")

Vemos que en los dataframes hay:

18 variables colineales en construccion

16 variables colineales en explotacion

Vamos a utilizar el código de la práctica 2 para eliminar las variables colineales de los dos dataframes.

In [ ]:
# Definimos cols y calculamos su rango
cols = list(construccion_mod.columns)
rank = np.linalg.matrix_rank(construccion_mod[cols].cov())
# Generamos un bucle while para que lo haga para cada columna
i = 0
while i < len(cols):
    col = cols[i]
    # Creamos una copia de las columnas sin la actual
    cols2 = cols.copy()
    cols2.remove(col)
    # Calculamos el rango sin esa columna
    construccion_mod_2 = construccion_mod[cols2]
    rank2 = np.linalg.matrix_rank(construccion_mod_2.cov())
    # Si el rango no cambia, eliminamos definitivamente la columna
    if rank == rank2:
        cols = cols2
        i = 0
    else:
        i += 1
        
# Ahora eliminamos las variables dependientes redefiniendo los 2 dataframes
construccion_mod = construccion_mod[cols]
explotacion_mod = explotacion_mod[cols]
# Comprobamos
print(construccion_mod.shape)
print(explotacion_mod.shape)
# Comprobamos el rango otra vez.
cov_explotacion = explotacion_mod.cov()
cov_construccion = construccion_mod.cov()
rango_explotacion = np.linalg.matrix_rank(cov_explotacion)
rango_construccion = np.linalg.matrix_rank(cov_construccion)
print(f"Test:{cov_explotacion.shape}, Train {cov_construccion.shape}")
print(f"Test {rango_explotacion}, Train {rango_construccion}")

Ahora quedan los dos datasets sin variables dependientes. Vemos que si se han eliminado correctamente las variables sobrantes y además podemos verificar como se han modificado. Ahora vamos a gestionar outliers y los valores nulos.

In [ ]:
for i in construccion:
    if construccion[[i]].isnull().sum()[i]> 0:
        print(f'CONSTRUCCIÓN Atributo "{i}", con valores Nulos {construccion[[i]].isnull().sum()[i]}')
        
for i in explotacion:
    if explotacion[[i]].isnull().sum()[i]> 0:
        print(f'EXPLOTACIÓN Atributo "{i}", con valores Nulos {explotacion[[i]].isnull().sum()[i]}')
        
for i in construccion_mod:
    if construccion_mod[[i]].isnull().sum()[i]> 0:
        print(f'CONSTRUCCIÓN_MOD Atributo "{i}", con valores Nulos {construccion_mod[[i]].isnull().sum()[i]}')
        
for i in explotacion_mod:
    if explotacion_mod[[i]].isnull().sum()[i]> 0:
        print(f'EXPLOTACIÓN_MOD Atributo "{i}", con valores Nulos {explotacion_mod[[i]].isnull().sum()[i]}')        

No hay valores nulos en los dataframes modificados, por tanto no nos preocupamos por los valores nulos. Solo hay valores nulos en los dataframes originales. Vamos a proseguir con los outliers. Para eliminar los outliers vamos a utilizar el método de las 3 sigmas o el z-score. Es un método seguro que nos asegura eliminar los outliers más alejados. No utilizamos el método del rango intercuantílico porque eliminariamos gran parte del dataframe.

In [ ]:
z_construccion = construccion_mod.apply(zscore)
z_explotacion = explotacion_mod.apply(zscore)
outliers_construccion = (z_construccion.abs() > 3)
outliers_explotacion = (z_explotacion.abs() > 3)
print(outliers_construccion.sum(),'\n')
print(outliers_explotacion.sum(),'\n')
print("Outliers",outliers_construccion.sum().sum())
print("Outliers",outliers_explotacion.sum().sum())


Vemos que hay pocos outliers, y todavía hay menos en el de explotación. Es normal, ya que hay muchos menos datos en el dataframe de explotación. Ahora vamos a proceder a su eliminación, utilizando el mismo método. Los hemos identificado con z-score. Y ahora vamos a eliminarlos con el método de los 3 sigmas. Además, vamos a eliminar los outliers del dataframe original para no tener problemas más adelante a la hora de entrenar los modelos.

In [ ]:
explotacion_mod = explotacion_mod[(z_explotacion.abs() < 3).all(axis=1)]
construccion_mod = construccion_mod[(z_construccion.abs() < 3).all(axis=1)]

# Eliminamos las columnas constantes para que no de problemas a la hora de entrenar modelos
columnas_constantes = construccion_mod.columns[construccion_mod.std() == 0].tolist()
if len(columnas_constantes) > 0:
    for col in columnas_constantes:
        construccion_mod = construccion_mod.drop(columns=columnas_constantes)
        explotacion_mod = explotacion_mod.drop(columns=columnas_constantes)

print(f"explotacion = {explotacion.shape}")
print(f"explotacion_mod = {explotacion_mod.shape}")
print(f"construccion = {construccion.shape}")
print(f"construccion_mod = {construccion_mod.shape}")

#### SEGUNDA PARTE
Ahora que hemos limpiado ambos dataframes, eliminando los outliers, gestionando los valores nulos y eliminando variables colineales vamos a proceder a construir un modelo dummy y un modelo de regresión lineal usando la librería scikit-klearn.

Pero antes de proseguir tenemos que aclarar de que no podemos tomar el dataframe explotación como el test, dado que su target son todos valores nulos no podemos verificarlo. Para ello, utilizando las librerías vamos a definir un dataframe nuevo llamado test que será generado a partir de construcción.

In [ ]:
# Definimos el eje y como el target y el eje x como el dataframe ya preprocesado
y_construccion = construccion.loc[construccion_mod.index, "Churn Status"]   
x_construccion = construccion_mod
x_explotacion = explotacion_mod
# IMPORTANTE: El dataset de construcción se divide en train y test
# El dataset de explotación se usa solo para predicción final
x_train, x_test, y_train, y_test = train_test_split(
    x_construccion, # Eje x del modelo
    y_construccion, # Eje y del modelo
    test_size=0.2, # Tomamos el 20% del dataset como test
    random_state=42,
    stratify=y_construccion)
# Generamos el modelo y lo entrenamos con el dataframe de construcción (dataframe train)
dummy_model = DummyClassifier()
dummy_model.fit(x_train, y_train)
y_pred_train = dummy_model.predict(x_train)
y_pred_test = dummy_model.predict(x_test)
# Las predicciones son arrays de numpy

Ahora, vamos a desarrollar la precisión de nuestro modelo con el R2 score y comprobar si tiene sobreajuste viendo la precisión de nuestro modelo sobre el dataset de construcción y  sobre el dataset de explotación.

In [ ]:
# Hallamos el coeficiente R2 de test y train
R2_dummy_train = dummy_model.score(x_train,y_train)
R2_dummy_test = dummy_model.score(x_test,y_test)
print(R2_dummy_train,R2_dummy_test)

El modelo dummy que hemos creado a base del datafram de construcción no tiene sobreajuste. Esto lo podemos ver en la poca diferencia que hay entre el R2_score de train y test. Ahora tenemos que pasar a analizar las predicciones que hace nuestro modelo.

In [ ]:
# Vamos a ver los valores únicos de nuestra base de datos original. 
# Tendrían que ser 0 o 1. También vamos a hallar el número de 1 y 0 y sus proporciones.
print(np.unique(construccion["Churn Status"]))
print(construccion["Churn Status"].value_counts())
print(construccion["Churn Status"].value_counts(normalize="True"))

Vemos que las proporciones tienden a que hay muchos más 0 (clientes no potenciales) que 1 (clientes no potenciales). Veamos ahora si nuestro modelo se corresponde a los valores originales:

In [ ]:
# Ahora vamos a centrarnos en los valores de predicción. 
# Tendrían que ser 0 o 1. También vamos a hallar el número de 1 y 0 y sus proporciones.
print(f"Valores únicos: {np.unique(y_pred_train)},{np.unique(y_pred_test)}")
print(f"Proporciones:{y_train.mode()[0]},{y_test.mode()[0]}")
print(f"Longitudes de los arrays: {len(y_pred_train)},{len(y_pred_test)}")
# Pasamos a utilizar predict_proba, vamos a centrarnos en el test que es el que nos interesa.
# Vamos a hallar las proporciones y los valores únicos nuevamente
y_proba_train = dummy_model.predict_proba(x_train)
y_proba_test = dummy_model.predict_proba(x_test)
print(f"Número de valores: {y_proba_train.shape},{y_proba_test.shape}")
print(f"Proporciones en train: {y_proba_train[0]} ")
print(f"Proporciones en test: {y_proba_test[0]}")

A la hora de hacer predicciones y compararlas con las de nuestros datos originales vemos que nuestro modelo dummy genera un array lleno de ceros. Esto se debe a que el target únicamente se compone de valores de cero o de uno y el más frecuente por una diferencia insignificante es el cero. El modelo dummy al evaluar por el dato más frecuente rellena de ceros la predicción para que sea más accurate. Al utilizar predict_proba() podemos ver que las estadísticas se acercan mucho más, casi igualandolas a los valores originales, concluyendo en un buen modelo.

#### TERCERA PARTE:

Ahora vamos a centrarnos en crear dos modelos de regresión logística con la librería sklearn utilizando LogisticRegression y LogisticRegressionCV. 

In [ ]:
# Creamos los modelos
LR_model = LogisticRegression()
LRCV_model = LogisticRegressionCV()
# Volvemos a utilizar x_train, x_test, y_train, y_test. 
# Tenemos que estandarizar x_train y x_test
x_train_scaled = x_train
x_test_scaled = x_test

for col in x_train.columns:
    media = x_train[col].mean()
    desviacion = x_train[col].std()
    x_train_scaled[col] = (x_train[col] - media) / desviacion
    x_test_scaled[col] = (x_test[col] - media) / desviacion

# Entrenamos los modelos
LR_model.fit(x_train_scaled, y_train)
LRCV_model.fit(x_train_scaled, y_train)

Hemos hecho un pequeño ajuste para que al entrenar el modelo no de error. Ahora bien, vamos a desarrollar las predicciones y vamos a comentar los resultados

In [ ]:
# Predicciones
y_pred_train_lr = LR_model.predict(x_train_scaled)
y_pred_test_lr = LR_model.predict(x_test_scaled)
y_pred_train_lrcv = LRCV_model.predict(x_train_scaled)
y_pred_test_lrcv = LRCV_model.predict(x_test_scaled)
# Scores
score_train_lr = LR_model.score(x_train_scaled, y_train)
score_test_lr = LR_model.score(x_test_scaled, y_test)
score_train_lrcv = LRCV_model.score(x_train_scaled, y_train)
score_test_lrcv = LRCV_model.score(x_test_scaled, y_test)

print(f"R2_Score de train LogisticRegression: {score_train_lr}")
print(f"R2_Score de test LogisticRegression: {score_test_lr}")
print(f"R2_Score de train LogisticRegressionCV: {score_train_lrcv}")
print(f"R2_Score de test LogisticRegressionCV: {score_test_lrcv}")

# Comparación con Dummy Classifier:
print(f"Mejora en train: {score_train_lr - R2_dummy_train}")
print(f"Mejora en test: {score_test_lr - R2_dummy_train}")

Comparando los dos modelos de regresión logística con el clasificador dummy es bastante mejor los modelos de regresión logística, ya que supera en veinte puntos porcentuales en train y quince puntos porcentuales en test. 

Por otra parte, viendo el R2_score de LogisticRegression y LogisticRegressionCV podemos concluir que no hay diferencia en los modelos. Muy probablemente podría haber una diferencia leve si modificáramos algunos parámetros del modelo. 

Para concluir, podemos observar que hay un sobreajuste mayor en estos dos modelos comparado con el Dummy pero sin embargo no es un sobreajuste notable, ya que es menos de cinco puntos porcentuales.